# MobileNetV2 mask classifier training

Install `requirements-dev.txt`, then provide an image dataset with `with_mask/`
and `without_mask/` folders using `MASK_DATASET_DIR`. The dataset is not bundled.
Running training replaces the bundled mask weights and plots with the new run.
The adjacent PNG curves are retained from the original project training run;
cleared notebook outputs do not claim a fresh training/evaluation result.

In [ ]:
# import the necessary packages
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, MobileNetV2
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import math

ROOT = Path.cwd()
if not (ROOT / "face_mask_detection").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "face_mask_detection").is_dir():
    raise RuntimeError("Open the notebook from the SmartParking repository.")
MODEL_DIR = ROOT / "face_mask_detection"

In [ ]:
#  Initial learning rate, epochs and batch size
INIT_LR = 1e-4
EPOCHS = 30
BS = 32

In [ ]:
DIRECTORY = Path(os.getenv("MASK_DATASET_DIR", str(MODEL_DIR / "dataset")))
if not all((DIRECTORY / label).is_dir() for label in ("with_mask", "without_mask")):
    raise FileNotFoundError("Set MASK_DATASET_DIR to a dataset with with_mask/ and without_mask/ folders.")
CATEGORIES = ["with_mask", "without_mask"]

In [ ]:
# Load image
print("[INFO] Loading images...")

data = []
labels = []

valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")

for category in CATEGORIES:
    path = os.path.join(DIRECTORY, category)
    for img_name in os.listdir(path):
        if img_name.lower().endswith(valid_extensions):
            img_path = os.path.join(path, img_name)
            try:
                image = load_img(img_path, target_size=(224, 224))
                image = img_to_array(image)
                image = preprocess_input(image)

                data.append(image)
                labels.append(category)
            except Exception as e:
                print(f"[WARNING] Không thể load ảnh {img_path}: {e}")


In [ ]:
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

data = np.array(data, dtype="float32")
labels = np.array(labels)


In [ ]:
(trainX, testX, trainY, testY) = train_test_split(
    data, labels, test_size=0.2, stratify=labels, random_state=42
)

In [ ]:
aug = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

In [ ]:
baseModel = MobileNetV2(weights="imagenet", include_top=False,
                        input_tensor=Input(shape=(224, 224, 3)))

# ============================ BUILD HEAD ============================
headModel = baseModel.output
headModel = AveragePooling2D(pool_size=(7, 7))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(128, activation="relu")(headModel)
headModel = Dropout(0.5)(headModel)
headModel = Dense(2, activation="softmax")(headModel)

model = Model(inputs=baseModel.input, outputs=headModel)

In [ ]:
# FREEZE BASE 
for layer in baseModel.layers:
    layer.trainable = False

# COMPILE MODEL 
print("[INFO] Compiling model...")
opt = Adam(learning_rate=INIT_LR)  # Updated keyword
model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["accuracy"])

In [ ]:
# TRAINING 
print("[INFO] Training model...")
# Define callback to save best model
checkpoint = ModelCheckpoint(
    str(MODEL_DIR / "mask_detector.h5"),    
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# Train the model with checkpoint callback
H = model.fit(
    aug.flow(trainX, trainY, batch_size=BS),
    steps_per_epoch=max(1, math.ceil(len(trainX) / BS)),
    validation_data=(testX, testY),
    epochs=EPOCHS,
    callbacks=[checkpoint]
)

In [ ]:
N = len(H.history["loss"])
plt.style.use("ggplot")

# Accuracy
plt.figure()
plt.plot(np.arange(0, N), H.history["accuracy"], label="train_acc")
plt.plot(np.arange(0, N), H.history["val_accuracy"], label="val_acc")  
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.savefig(MODEL_DIR / "accuracy_plot.png")
plt.close()

# Loss
plt.figure()
plt.plot(np.arange(0, N), H.history["loss"], label="train_loss")
plt.plot(np.arange(0, N), H.history["val_loss"], label="val_loss")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(loc="upper right")
plt.savefig(MODEL_DIR / "loss_plot.png")
plt.close()